In [ ]:
import glob
import sys
from pathlib import Path

BASE_FOLDER = Path.cwd().parent
sys.path.append(str(BASE_FOLDER))

import pandas as pd
import wandb

from src.trainers.utils import compute_final_metrics

data_folder = Path("/home/hkristen/Nextcloud/HabitAlp2.0/Originaldaten")

# Config

In [ ]:
# Ground truth reference file paths
reference_file_path_2013 = data_folder / "processed/mask/classes_v4_2013.tif"
reference_file_path_2020 = data_folder / "processed/mask/classes_v4_2020.tif"
reference_file_path_2013_test_cells = data_folder / "processed/mask/classes_v4_2013_test_cells.tif"

# Wandb configuration
wandb_project = "semantic-segmentation-terratorch-extended"

# Mapping of experiment names to their corresponding wandb IDs
wandb_experiments = {                                                                                                                                                                                                                                                                 
      "clay-v1.5-rgb-nir-ndsm-phaseE": "80dh36vg",
      "clay-v1.5-rgb-nir-phaseF": "jjltacr4",                                                                                                                                                                                                                                                 
      "clay-v1-rgb-nir-ndsm-phaseE": "dkggiaj1",                                                                                                                                                                                                                                              
      "clay-v1-rgb-nir-phaseF": "xkgkd5hp",
      "dofa-base-rgb-nir-ndsm-phaseE": "bho779er",                                                                                                                                                                                                                                            
      "dofa-base-rgb-nir-phaseF": "ap0j4q56",                                                                                                                                                                                                                                                 
      "prithvi-v2-300-rgb-nir-ndsm-phaseE": "62ol6euh",
      "prithvi-v2-300-rgb-nir-phaseF": "74qyfxdx",                                                                                                                                                                                                                                            
      "terramind-rgb-nir-ndsm-phaseE": "ba0nku12",                                                                                                                                                                                                                                            
      "terramind-rgb-nir-phaseF": "s5yjepv2",
      "unet-mit-b2-rgb-nir-ndsm-phaseE":  "eih9va5p",                                                                                                                                                                                                                                          
      "unet-mit-b2-rgb-nir-phaseF": "6cayufzk",
      "clay-v1.5-rgb-phaseG": "k3i2agz3",
      "dofa-base-rgb-phaseG": "46t5sv6p",
      "terramind-rgb-phaseG": "ybqw4nwa",
}

# Batch-Wise Classification Metrics before Post-Processing

In [ ]:
for experiment, wandb_id in wandb_experiments.items():
    model_output_folder = data_folder / "model_output/" / experiment

    with wandb.init(project=wandb_project, id=wandb_id, resume="must") as run:
        # Find the prediction file in the model output folder
        prediction_files_2013 = glob.glob(str(model_output_folder) + "/2013_*.tif")
        if prediction_files_2013:
            prediction_2013_path = Path(prediction_files_2013[0])
            print("Prediction 2013 File: ", prediction_2013_path.name)

            multiclass_metrics, metrics_each_label, figure_collection = compute_final_metrics(
                reference_file_path=reference_file_path_2013_test_cells,
                prediction_file_path=prediction_2013_path,
                num_classes=24, # 24 classes to take index into account
                ignore_index=0,
            )
            metrics_each_label.pop("MulticlassConfusionMatrix")

            for metric, value in multiclass_metrics.items():
                print(f"{metric}: {round(float(value), 3)}")

            # Save table in experiment folder
            metrics_df = pd.DataFrame([multiclass_metrics], index=["macro"]).astype(float)
            metrics_each_label_df = pd.DataFrame(metrics_each_label)
            metrics_df = pd.concat([metrics_df, metrics_each_label_df])
            metrics_df.to_csv(model_output_folder / "classification_metrics_2013.csv")

            # Log metrics to wandb
            run.summary["classification_metrics_2013"] = multiclass_metrics
            change_metrics_each_class_table = wandb.Table(dataframe=metrics_each_label_df)
            run.log({"classification_metrics_2013/table_each_class": change_metrics_each_class_table})
            run.log({"classification_metrics_2013/confusion_matrix": wandb.Image(figure_collection[0])})

        prediction_files_2020 = glob.glob(str(model_output_folder) + "/2020_*.tif")
        if prediction_files_2020:
            prediction_2020_path = Path(prediction_files_2020[0])
            print("Prediction 2020 File: ", prediction_2020_path.name)

            multiclass_metrics, metrics_each_label, figure_collection = compute_final_metrics(
                reference_file_path=reference_file_path_2020,
                prediction_file_path=prediction_2020_path,
                num_classes=24, # 24 classes to take index into account
                ignore_index=0,
            )
            metrics_each_label.pop("MulticlassConfusionMatrix")

            for metric, value in multiclass_metrics.items():
                print(f"{metric}: {round(float(value), 3)}")

            # Save table in experiment folder
            metrics_df = pd.DataFrame([multiclass_metrics], index=["macro"]).astype(float)
            metrics_each_label_df = pd.DataFrame(metrics_each_label)
            metrics_df = pd.concat([metrics_df, metrics_each_label_df])
            metrics_df.to_csv(model_output_folder / "classification_metrics_2020.csv")

            # Log metrics to wandb
            run.summary["classification_metrics_2020"] = multiclass_metrics
            change_metrics_each_class_table = wandb.Table(dataframe=metrics_each_label_df)
            run.log({"classification_metrics_2020/table_each_class": change_metrics_each_class_table})
            run.log({"classification_metrics_2020/confusion_matrix": wandb.Image(figure_collection[0])})

# Batch-Wise Change Map Generation

In [ ]:
from src.inference.eval import generate_change_map

for experiment, wandb_id in wandb_experiments.items():
    model_output_folder = data_folder / "model_output/" / experiment
    prediction_change_map_2013_2020 = model_output_folder / "change_map_2013-2020.tif"

    prediction_files_2020 = glob.glob(str(model_output_folder) + "/2020_*.tif")
    if prediction_files_2020:
        prediction_2020_path = Path(prediction_files_2020[0])
        print("Prediction 2020 File: ", prediction_2020_path.name)

        generate_change_map(
            mask_path=reference_file_path_2013,
            prediction_path=prediction_2020_path,
            output_path=prediction_change_map_2013_2020,
        )

# Batch-Wise Change Metrics before Postprocessing

## Multi-Class

In [ ]:
reference_change_map_2013_2020 = data_folder / "habitalp_change/habitalp_change_v4_2013-2020.tif"

class_names = [
    "No change",
    "Mature Tree Density Loss",
    "Old Growth Density Loss",
    "Forest Setback YoungLoss",
    "Forest Stage Progression",
    "Forest Density Gain",
    "Early Forest Establishment",
    "Clearcut Loss",
    "Other Transition"
]

for experiment, wandb_id in wandb_experiments.items():
    model_output_folder = data_folder / "model_output/" / experiment
    prediction_change_map_2013_2020 = model_output_folder / "change_map_2013-2020.tif"

    with wandb.init(project=wandb_project, id=wandb_id, resume="must") as run:
        # Multiclass 
        multiclass_metrics, metrics_each_label, figure_collection = compute_final_metrics(
            reference_file_path= reference_change_map_2013_2020,
            prediction_file_path=prediction_change_map_2013_2020,
            num_classes=9, # 9 classes to take no change and index into account
            class_names=class_names,
            ignore_index=255,
        )
        metrics_each_label.pop("MulticlassConfusionMatrix")

        for metric, value in multiclass_metrics.items():
            print(f"{metric}: {round(float(value), 3)}")

        # Save table in experiment folder
        metrics_df = pd.DataFrame([multiclass_metrics], index=["macro"]).astype(float)
        metrics_each_label_df = pd.DataFrame(metrics_each_label)
        metrics_df = pd.concat([metrics_df, metrics_each_label_df])
        metrics_df.to_csv(model_output_folder / "change_metrics_multiclass_2013-2020.csv")

        # Log metrics to wandb
        run.summary["change_metrics_multiclass_2013-2020"] = multiclass_metrics
        change_metrics_each_class_table = wandb.Table(dataframe=metrics_each_label_df)
        run.log({"change_metrics_multiclass_2013-2020/table_each_class": change_metrics_each_class_table})
        run.log({"change_metrics_multiclass_2013-2020/confusion_matrix": wandb.Image(figure_collection[0])})

## Binary

In [ ]:
reference_change_map_2013_2020 = data_folder / "habitalp_change/habitalp_change_v4_2013-2020.tif"

class_names = [
    "No change",
    "Change"
]

for experiment, wandb_id in wandb_experiments.items():
    model_output_folder = data_folder / "model_output/" / experiment
    prediction_change_map_2013_2020 = model_output_folder / "change_map_2013-2020.tif"

    with wandb.init(project=wandb_project, id=wandb_id, resume="must") as run:
        # Binary 
        multiclass_metrics, metrics_each_label, figure_collection = compute_final_metrics(
            reference_file_path= reference_change_map_2013_2020,
            prediction_file_path=prediction_change_map_2013_2020,
            num_classes=2,
            class_names=class_names,
            ignore_index=255,
        )
        metrics_each_label.pop("MulticlassConfusionMatrix")

        for metric, value in multiclass_metrics.items():
            print(f"{metric}: {round(float(value), 3)}")

        # Save table in experiment folder
        metrics_df = pd.DataFrame([multiclass_metrics], index=["macro"]).astype(float)
        metrics_each_label_df = pd.DataFrame(metrics_each_label)
        metrics_df = pd.concat([metrics_df, metrics_each_label_df])
        metrics_df.to_csv(model_output_folder / "change_metrics_binary_2013-2020.csv")

        # Log metrics to wandb
        run.summary["change_metrics_binary_2013-2020"] = multiclass_metrics
        change_metrics_each_class_table = wandb.Table(dataframe=metrics_each_label_df)
        run.log({"change_metrics_binary_2013-2020/table_each_class": change_metrics_each_class_table})
        run.log({"change_metrics_binary_2013-2020/confusion_matrix": wandb.Image(figure_collection[0])})

# Re-Rasterize GeoPackage

In [7]:
import geopandas as gpd
import rasterio as rio
from rasterio.features import rasterize

# Use existing raster file as reference for pixel-matching output
reference_label_file = rio.open(reference_file_path_2020)

out_transform = reference_label_file.transform
out_width = reference_label_file.width
out_height = reference_label_file.height
nodata_value = reference_label_file.nodata

for year in [2013, 2020]:
    for experiment, _ in wandb_experiments.items():
        model_output_folder = data_folder / "model_output/" / experiment

        vectorized_result_file_path = model_output_folder / f"constraint_resolution_+_polygonfix_cc_updated/prediction_{year}_constraint_resolution_cc_updated.gpkg"
        prediction_postprocessed_path = model_output_folder / f"constraint_resolution_+_polygonfix_cc_updated/{year}_re-rasterized.tif"

        if not vectorized_result_file_path.exists():
            raise FileNotFoundError(f"Vectorized result file not found at {vectorized_result_file_path}")

        vectorized_result = gpd.read_file(
            vectorized_result_file_path
        )

        # this is where we create a generator of geom, value pairs to use in rasterizing
        shapes = (
            (geom, value)
            for geom, value in zip(vectorized_result.geometry, vectorized_result["Class"])
        )

        rasterized_result = rasterize(
            shapes=shapes,
            out_shape=(out_height, out_width),
            transform=out_transform,
            fill=nodata_value,
            nodata=nodata_value,
            dtype=rio.uint8,
            default_value=99,
        )

        # Save cropped raster
        with rio.open(
            prediction_postprocessed_path,
            "w",
            driver="GTiff",
            width=out_width,
            height=out_height,
            count=1,
            crs=reference_label_file.crs,
            transform=out_transform,
            dtype=rio.uint8,
            nodata=nodata_value,
            compress="lzw",
            tiled=True,
            predictor=2,
            bigtiff=True,
        ) as dst:
            dst.write(rasterized_result, indexes=1)

# Batch-Wise Classification Metrics after Post-Processing

In [ ]:
for experiment, wandb_id in wandb_experiments.items():
    model_output_folder = data_folder / "model_output" / experiment / "constraint_resolution_+_polygonfix_cc_updated"

    with wandb.init(project=wandb_project, id=wandb_id, resume="must") as run:
        # Find the prediction file in the model output folder
        prediction_files_2013 = glob.glob(str(model_output_folder) + "/2013_re-rasterized.tif")
        if prediction_files_2013:
            prediction_2013_path = Path(prediction_files_2013[0])
            print("Prediction 2013 File: ", prediction_2013_path.name)

            multiclass_metrics, metrics_each_label, figure_collection = compute_final_metrics(
                reference_file_path=reference_file_path_2013_test_cells,
                prediction_file_path=prediction_2013_path,
                num_classes=24, # 24 classes to take index into account
                ignore_index=0,
            )
            metrics_each_label.pop("MulticlassConfusionMatrix")

            for metric, value in multiclass_metrics.items():
                print(f"{metric}: {round(float(value), 3)}")

            # Save table in experiment folder
            metrics_df = pd.DataFrame([multiclass_metrics], index=["macro"]).astype(float)
            metrics_each_label_df = pd.DataFrame(metrics_each_label)
            metrics_df = pd.concat([metrics_df, metrics_each_label_df])
            metrics_df.to_csv(model_output_folder / "classification_metrics_2013_after_postprocessing.csv")

            # Log metrics to wandb
            run.summary["classification_metrics_2013_after_postprocessing"] = multiclass_metrics
            change_metrics_each_class_table = wandb.Table(dataframe=metrics_each_label_df)
            run.log({"classification_metrics_2013/table_each_class_after_postprocessing": change_metrics_each_class_table})
            run.log({"classification_metrics_2013/confusion_matrix_after_postprocessing": wandb.Image(figure_collection[0])})

        prediction_files_2020 = glob.glob(str(model_output_folder) + "/2020_re-rasterized.tif")
        if prediction_files_2020:
            prediction_2020_path = Path(prediction_files_2020[0])
            print("Prediction 2020 File: ", prediction_2020_path.name)

            multiclass_metrics, metrics_each_label, figure_collection = compute_final_metrics(
                reference_file_path=reference_file_path_2020,
                prediction_file_path=prediction_2020_path,
                num_classes=24, # 24 classes to take index into account
                ignore_index=0,
            )
            metrics_each_label.pop("MulticlassConfusionMatrix")

            for metric, value in multiclass_metrics.items():
                print(f"{metric}: {round(float(value), 3)}")

            # Save table in experiment folder
            metrics_df = pd.DataFrame([multiclass_metrics], index=["macro"]).astype(float)
            metrics_each_label_df = pd.DataFrame(metrics_each_label)
            metrics_df = pd.concat([metrics_df, metrics_each_label_df])
            metrics_df.to_csv(model_output_folder / "classification_metrics_2020_after_postprocessing.csv")

            # Log metrics to wandb
            run.summary["classification_metrics_2020_after_postprocessing"] = multiclass_metrics
            change_metrics_each_class_table = wandb.Table(dataframe=metrics_each_label_df)
            run.log({"classification_metrics_2020/table_each_class_after_postprocessing": change_metrics_each_class_table})
            run.log({"classification_metrics_2020/confusion_matrix_after_postprocessing": wandb.Image(figure_collection[0])})

# Batch-Wise Change Map Generation after Postprocessing

In [ ]:
from src.inference.eval import generate_change_map

for experiment, wandb_id in wandb_experiments.items():
    model_output_folder = data_folder / "model_output" / experiment / "constraint_resolution_+_polygonfix_cc_updated"
    prediction_change_map_2013_2020 = model_output_folder / "change_map_2013-2020.tif"

    prediction_files_2020 = glob.glob(str(model_output_folder) + "/2020_re-rasterized.tif")
    if prediction_files_2020:
        prediction_2020_path = Path(prediction_files_2020[0])
        print("Prediction 2020 File: ", prediction_2020_path.name)

        generate_change_map(
            mask_path=reference_file_path_2013,
            prediction_path=prediction_2020_path,
            output_path=prediction_change_map_2013_2020,
        )

# Batch-Wise Change Metrics after Postprocessing

## Multi-Class

In [ ]:
reference_change_map_2013_2020 = data_folder / "habitalp_change/habitalp_change_v4_2013-2020.tif"

class_names = [
    "No change",
    "Mature Tree Density Loss",
    "Old Growth Density Loss",
    "Forest Setback YoungLoss",
    "Forest Stage Progression",
    "Forest Density Gain",
    "Early Forest Establishment",
    "Clearcut Loss",
    "Other Transition"
]

for experiment, wandb_id in wandb_experiments.items():
    model_output_folder = data_folder / "model_output/" / experiment / "constraint_resolution_+_polygonfix_cc_updated"
    prediction_change_map_2013_2020 = model_output_folder / "change_map_2013-2020.tif"

    with wandb.init(project=wandb_project, id=wandb_id, resume="must") as run:
        # Multiclass 
        multiclass_metrics, metrics_each_label, figure_collection = compute_final_metrics(
            reference_file_path= reference_change_map_2013_2020,
            prediction_file_path=prediction_change_map_2013_2020,
            num_classes=9, # 9 classes to take no change and index into account
            class_names=class_names,
            ignore_index=255,
        )
        metrics_each_label.pop("MulticlassConfusionMatrix")

        for metric, value in multiclass_metrics.items():
            print(f"{metric}: {round(float(value), 3)}")

        # Save table in experiment folder
        metrics_df = pd.DataFrame([multiclass_metrics], index=["macro"]).astype(float)
        metrics_each_label_df = pd.DataFrame(metrics_each_label)
        metrics_df = pd.concat([metrics_df, metrics_each_label_df])
        metrics_df.to_csv(model_output_folder / "change_metrics_multiclass_2013-2020_after_postprocessing.csv")

        # Log metrics to wandb
        run.summary["change_metrics_multiclass_2013-2020_after_postprocessing"] = multiclass_metrics
        change_metrics_each_class_table = wandb.Table(dataframe=metrics_each_label_df)
        run.log({"change_metrics_multiclass_2013-2020/table_each_class_after_postprocessing": change_metrics_each_class_table})
        run.log({"change_metrics_multiclass_2013-2020/confusion_matrix_after_postprocessing": wandb.Image(figure_collection[0])})

## Binary

In [ ]:
reference_change_map_2013_2020 = data_folder / "habitalp_change/habitalp_change_v4_2013-2020.tif"

class_names = [
    "No change",
    "Change"
]

for experiment, wandb_id in wandb_experiments.items():
    model_output_folder = data_folder / "model_output/" / experiment / "constraint_resolution_+_polygonfix_cc_updated"
    prediction_change_map_2013_2020 = model_output_folder / "change_map_2013-2020.tif"

    with wandb.init(project=wandb_project, id=wandb_id, resume="must") as run:
        # Binary 
        multiclass_metrics, metrics_each_label, figure_collection = compute_final_metrics(
            reference_file_path= reference_change_map_2013_2020,
            prediction_file_path=prediction_change_map_2013_2020,
            num_classes=2,
            class_names=class_names,
            ignore_index=255,
        )
        metrics_each_label.pop("MulticlassConfusionMatrix")

        for metric, value in multiclass_metrics.items():
            print(f"{metric}: {round(float(value), 3)}")

        # Save table in experiment folder
        metrics_df = pd.DataFrame([multiclass_metrics], index=["macro"]).astype(float)
        metrics_each_label_df = pd.DataFrame(metrics_each_label)
        metrics_df = pd.concat([metrics_df, metrics_each_label_df])
        metrics_df.to_csv(model_output_folder / "change_metrics_binary_2013-2020_after_postprocessing.csv")

        # Log metrics to wandb
        run.summary["change_metrics_binary_2013-2020_after_postprocessing"] = multiclass_metrics
        change_metrics_each_class_table = wandb.Table(dataframe=metrics_each_label_df)
        run.log({"change_metrics_binary_2013-2020/table_each_class_after_postprocessing": change_metrics_each_class_table})
        run.log({"change_metrics_binary_2013-2020/confusion_matrix_after_postprocessing": wandb.Image(figure_collection[0])})